# Adaptive BubbleMaster scan planner

Generate midpoint-convention BubbleMaster setup files and ordinary Perlmutter CPU/GPU SLURM jobs. The production-domain assumptions are fixed to $\gamma_{ij,\max}=2.5\gamma_{*,\max}$ and $t_{\max}=1.4R_*(\gamma_{*,\max})$. The frequency grid is non-rectangular and covers the requested $kR_*$ interval only where each pair gamma can occur.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import h5py
import numpy as np
from scipy.optimize import brentq

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams, write_2d_setup
from ptbuilder.weight_scan_diagnostics import load_kinematics

ROOT = Path(pt.__file__).resolve().parent.parent


## Physics-domain inputs

These are the inputs that define the physical scan domain.

In [ ]:
KR_MIN = 1.0
KR_MAX = 100.0
N_OMEGA = 5
LAMBDA_BAR = 0.84
GAMMA_STAR_MIN = 1.0
GAMMA_STAR_MAX = 3.2


## Numerical and scheduler controls

These do not change the intended physical domain, but must be converged separately. One ordinary multi-node SLURM job launches one collision-gamma slice per node as concurrent exclusive job steps; no job array is used.

In [ ]:
N_GAMMA_IJ = 5         # pair-gamma samples across the required range
N_TIME = 5             # cutoff-time samples per pair gamma
DZ = 0.005
N_K = 51
HOW_OFTEN_DS = 5
BABY_STEPS = 100
CPU_WALLTIME = '04:00:00'
GPU_WALLTIME = '04:00:00'
GPU_ACCOUNT = 'm3166_g'

PAIR_GAMMA_FACTOR = 2.5
TIME_FACTOR = 1.4


In [ ]:
if not (0 < KR_MIN < KR_MAX):
    raise ValueError('Require 0 < KR_MIN < KR_MAX')
if not (1 <= GAMMA_STAR_MIN <= GAMMA_STAR_MAX):
    raise ValueError('Require 1 <= GAMMA_STAR_MIN <= GAMMA_STAR_MAX')
if N_OMEGA < 2 or N_TIME < 2 or D_GAMMA_IJ <= 0:
    raise ValueError('Invalid numerical grid controls')

instanton_path = ROOT / f'data/phi4_lambda_bar{LAMBDA_BAR:g}/instanton.h5'
kin = load_kinematics(instanton_path)

class CachedPhi4Potential:
    def __init__(self, lambda_bar):
        self.params = {'lambda_bar': float(lambda_bar)}
    def to_hdf5(self, group):
        group.attrs.create('type', 'phi4', dtype=h5py.string_dtype(encoding='ascii'))
        group.attrs['phi_false'] = 0.0
        group.attrs['phi_true'] = 1.0
        group.attrs['lambda_bar'] = self.params['lambda_bar']

model = SimpleNamespace(instanton=kin.profile,
                        kinematics=kin,
                        potential=CachedPhi4Potential(LAMBDA_BAR))

def time_at_gamma(gamma):
    if gamma <= 1.0:
        return 0.0
    return brentq(lambda t: float(kin.Gamma(t)) - gamma, 0.0, 1.0e7)

def rstar_at_gamma_star(gamma_star):
    t_star = time_at_gamma(gamma_star)
    return 2.0 * float(kin.R(t_star, r='mid'))

RSTAR_MIN = rstar_at_gamma_star(GAMMA_STAR_MIN)
RSTAR_MAX = rstar_at_gamma_star(GAMMA_STAR_MAX)
PAIR_GAMMA_MAX = PAIR_GAMMA_FACTOR * GAMMA_STAR_MAX
T_MAX = TIME_FACTOR * RSTAR_MAX
OMEGA_MIN = KR_MIN / RSTAR_MAX

pair_gammas = np.linspace(1.0, PAIR_GAMMA_MAX, N_GAMMA_IJ)

print(f'R* range: {RSTAR_MIN:.6g} .. {RSTAR_MAX:.6g}')
print(f'pair gamma: {pair_gammas[0]:.6g} .. {pair_gammas[-1]:.6g} ({len(pair_gammas)} slices)')
print(f'global omega_min: {OMEGA_MIN:.6g}')
print(f'global t_max: {T_MAX:.6g}')


For a fixed pair gamma, the smallest target scale in which it is retained is $\max(\gamma_{*,\min},\gamma_{ij}/2.5)$. This sets its largest required $\omega$. Low-frequency coverage is set globally by the largest target $R_*$. The first requested cutoff time is the pair's midpoint collision time; BubbleMaster still accumulates the earlier integral into that first result.

In [ ]:
tag = (f'lb{LAMBDA_BAR:g}_gs{GAMMA_STAR_MIN:g}-{GAMMA_STAR_MAX:g}'
       f'_kR{KR_MIN:g}-{KR_MAX:g}')
scan_root = ROOT / 'data' / f'adaptive_scan_{tag}'
setup_dir = scan_root / 'setups'
setup_dir.mkdir(parents=True, exist_ok=True)
(scan_root / 'cpu').mkdir(exist_ok=True)
(scan_root / 'gpu').mkdir(exist_ok=True)
(ROOT / 'logs').mkdir(exist_ok=True)

params = BubbleMasterParams(
    dz=DZ, n_w=N_OMEGA, n_k=N_K, how_often_ds=HOW_OFTEN_DS,
    baby_steps=BABY_STEPS, cutoff_type=0, t_0_scal=1.0,
    collision_radius='mid',
)

manifest_rows = []
plan_rows = []
for index, gamma_ij in enumerate(pair_gammas):
    smallest_target = max(GAMMA_STAR_MIN, gamma_ij / PAIR_GAMMA_FACTOR)
    omega_max = KR_MAX / rstar_at_gamma_star(smallest_target)
    if omega_max <= OMEGA_MIN:
        raise RuntimeError(f'Empty omega range at gamma_ij={gamma_ij}')
    omega = np.geomspace(OMEGA_MIN, omega_max, N_OMEGA)
    t_first = time_at_gamma(gamma_ij)
    if t_first >= T_MAX:
        raise RuntimeError(f'Collision time exceeds scan end at gamma_ij={gamma_ij}')
    times = np.linspace(t_first, T_MAX, N_TIME)
    name = f'g{gamma_ij:08.3f}'.replace('.', 'p')
    setup = setup_dir / f'{name}.h5'
    write_2d_setup(model, gamma_ij, times, setup, params, wlist=omega)
    with h5py.File(setup, 'a') as handle:
        handle.attrs['target_gamma_star_min'] = GAMMA_STAR_MIN
        handle.attrs['target_gamma_star_max'] = GAMMA_STAR_MAX
        handle.attrs['kR_min'] = KR_MIN
        handle.attrs['kR_max'] = KR_MAX
        handle.attrs['pair_gamma_factor'] = PAIR_GAMMA_FACTOR
        handle.attrs['time_factor'] = TIME_FACTOR
        handle.attrs['R_star_max'] = RSTAR_MAX
    manifest_rows.append((index, gamma_ij, setup.relative_to(ROOT), name))
    plan_rows.append((gamma_ij, t_first, T_MAX, OMEGA_MIN, omega_max))

manifest = scan_root / 'manifest.tsv'
manifest.write_text(
    'index\tgamma_ij\tsetup\tname\n' +
    ''.join(f'{i}\t{g:.12g}\t{s}\t{n}\n' for i, g, s, n in manifest_rows)
)
np.savetxt(scan_root / 'scan_plan.txt', np.asarray(plan_rows),
           header='gamma_ij t_first t_max omega_min omega_max')
print(f'Wrote {len(manifest_rows)} setups under {setup_dir}')


In [ ]:
relative_manifest = manifest.relative_to(ROOT)
n_scan_nodes = len(manifest_rows)
cpu_script = ROOT / 'scripts' / f'adaptive_scan_{tag}_cpu.sh'
gpu_script = ROOT / 'scripts' / f'adaptive_scan_{tag}_gpu.sh'

cpu_text = f'''#!/bin/bash
#SBATCH --job-name=bm_{tag}_cpu
#SBATCH --output=logs/bm_{tag}_cpu_%j.out
#SBATCH --constraint=cpu
#SBATCH --qos=premium
#SBATCH --nodes={n_scan_nodes}
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=128
#SBATCH --time={CPU_WALLTIME}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"
export SLURM_CPU_BIND=cores
export OMP_PLACES=threads
export OMP_PROC_BIND=spread
export OMP_NUM_THREADS=128
export BM_OMEGA_TIMINGS=1
pids=()
while IFS=$'\t' read -r index gamma setup name; do
    [[ "$index" == "index" ]] && continue
    output="data/adaptive_scan_{tag}/cpu/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 128 --cpu-bind=cores bin/bubblemaster "$setup" "$output/" &
    pids+=("$!")
done < {relative_manifest}
status=0
for pid in "${{pids[@]}}"; do
    wait "$pid" || status=1
done
exit "$status"
'''

gpu_text = f'''#!/bin/bash
#SBATCH --job-name=bm_{tag}_gpu
#SBATCH --output=logs/bm_{tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {GPU_ACCOUNT}
#SBATCH --nodes={n_scan_nodes}
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={GPU_WALLTIME}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"
pids=()
while IFS=$'\t' read -r index gamma setup name; do
    [[ "$index" == "index" ]] && continue
    output="data/adaptive_scan_{tag}/gpu/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores --gpu-bind=single:1 bin/bubblemaster_gpu "$setup" "$output/" &
    pids+=("$!")
done < {relative_manifest}
status=0
for pid in "${{pids[@]}}"; do
    wait "$pid" || status=1
done
exit "$status"
'''
cpu_script.write_text(cpu_text)
gpu_script.write_text(gpu_text)
cpu_script.chmod(0o755)
gpu_script.chmod(0o755)
print(cpu_script)
print(gpu_script)
print(f'CPU: sbatch {cpu_script.relative_to(ROOT)}')
print(f'GPU: sbatch {gpu_script.relative_to(ROOT)}')
print('Submit only one backend unless intentionally benchmarking both.')


## Notes for loading the result

Each collision-gamma slice has its own frequency grid and starts at its own collision time. The older rectangular `RegularGridInterpolator` helpers assume common gamma/time/frequency grids and must not be used unchanged. The reconstruction loader must interpolate each slice only within its stored domain and reject out-of-domain requests rather than clipping or silently filling them.